# Solución · Lab 05 · Ordenar y limpiar

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/crapdev/sparkLab/blob/main/labs/05-ordenar-limpiar/solution.ipynb)

**Objetivo:** Dejar un DataFrame ordenado y sin basura: sort, limit, drop, duplicados con dropDuplicates y distinct, y nulos con dropna y fillna.

**Tiempo estimado:** 40 minutos

> Esta es la solución completa. Intenta resolver el laboratorio por tu cuenta antes
> de leerla: equivocarse y corregir es donde más se aprende.

## Preparación

Esta celda instala la versión de PySpark con la que se probó el laboratorio y comprueba
que hay una versión de Java compatible. En Colab tarda alrededor de un minuto la primera vez.

In [ ]:
import importlib.metadata
import os
import re
import shutil
import subprocess
import sys

PYSPARK_VERSION = "4.2.0"
MIN_JAVA_VERSION = 17


def installed_pyspark_version():
    try:
        return importlib.metadata.version("pyspark")
    except importlib.metadata.PackageNotFoundError:
        return None


def java_major_version():
    java = shutil.which("java")
    if java is None:
        return 0
    output = subprocess.run([java, "-version"], capture_output=True, text=True).stderr
    match = re.search(r'version "(\d+)', output)
    return int(match.group(1)) if match else 0


if installed_pyspark_version() != PYSPARK_VERSION:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", f"pyspark=={PYSPARK_VERSION}"],
        check=True,
    )

if java_major_version() < MIN_JAVA_VERSION:
    if "google.colab" not in sys.modules:
        raise RuntimeError("Spark 4 necesita Java 17 o superior. Instálalo y repite esta celda.")
    subprocess.run(["apt-get", "install", "-y", "-qq", "openjdk-17-jdk-headless"], check=True)
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

print(f"PySpark {PYSPARK_VERSION} y Java {java_major_version()} listos.")

La siguiente celda define `check`, la función que usan los tests. No hace falta que la
modifiques: ejecútala y sigue.

In [ ]:
RESULTS = {}


def check(name, condition, hint):
    """Run one test: `condition` is a function that must return True."""
    try:
        passed = bool(condition())
        error = None
    except Exception as exc:  # A broken answer is a failed test, not a crash.
        passed, error = False, exc
    RESULTS[name] = passed
    if passed:
        print(f"✅ {name}")
        return
    detail = f" ({type(error).__name__}: {error})" if error else ""
    print(f"❌ {name}{detail}")
    print(f"   Pista: {hint}")
    assert passed, f"{name} · {hint}"

## Sesión de Spark

Esta celda crea la `SparkSession` del laboratorio e importa las funciones como `F`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.master("local[*]").appName("lab-05-ordenar-limpiar").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print(f"Spark {spark.version} listo.")

## Datos

Esta celda descarga `bigmart_sales.csv` la primera vez y los deja en la carpeta `data/`.
Usa `dataset_path("nombre")` para obtener la ruta de un archivo.

In [ ]:
import os
import urllib.request

DATA_BASE_URLS = [
    "https://raw.githubusercontent.com/crapdev/sparkLab/main/public/data",
]
DATA_DIR = os.environ.get("PYSPARK_LABS_DATA_DIR", "data")
DATASETS = ["bigmart_sales.csv"]


def dataset_path(name):
    """Return the local path of a dataset, downloading it the first time."""
    path = os.path.join(DATA_DIR, name)
    if os.path.exists(path):
        return path
    os.makedirs(DATA_DIR, exist_ok=True)
    for base_url in DATA_BASE_URLS:
        try:
            urllib.request.urlretrieve(f"{base_url}/{name}", path)
            return path
        except OSError:
            continue  # Try the next mirror.
    raise RuntimeError(f"No se pudo descargar {name}. Revisa tu conexión y repite la celda.")


for dataset in DATASETS:
    print("Listo:", dataset_path(dataset))

In [ ]:
sales = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(dataset_path("bigmart_sales.csv"))
)

print("Filas:", sales.count(), "· Columnas:", len(sales.columns))

## Tarea 1 · Ordenar

`sort` y `orderBy` son la misma función. Reciben una o varias columnas y, para cada una,
el sentido: `F.col("x").asc()` (el de por defecto) o `F.col("x").desc()`.

Crea dos DataFrames a partir de `sales`:

- `heaviest`: ordenado por `Item_Weight` de mayor a menor
- `by_two`: ordenado por `Item_Weight` de mayor a menor y, a igual peso, por
  `Item_Visibility` de menor a mayor

In [ ]:
heaviest = sales.sort(F.col("Item_Weight").desc())
by_two = sales.sort(F.col("Item_Weight").desc(), F.col("Item_Visibility").asc())

In [ ]:
check(
    "Tarea 1 · heaviest empieza por el peso máximo, 21.35",
    lambda: heaviest.first()["Item_Weight"] == 21.35,
    'Usa sales.sort(F.col("Item_Weight").desc()). Si la primera fila tiene el peso en NULL, has ordenado ascendente.',
)
check(
    "Tarea 1 · heaviest conserva las 8523 filas y 12 columnas",
    lambda: heaviest.count() == 8523 and len(heaviest.columns) == 12,
    "Ordenar no quita filas ni columnas.",
)
check(
    "Tarea 1 · by_two desempata por visibilidad ascendente",
    lambda: by_two.first()["Item_Weight"] == 21.35 and by_two.first()["Item_Visibility"] == 0.068765205,
    'La segunda clave es F.col("Item_Visibility").asc(). Entre los 7 productos de 21.35, el de menor visibilidad tiene 0.068765205.',
)

Mira dónde quedan los nulos. `Item_Weight` tiene 1463 valores nulos: en orden ascendente
salen **primero** y en descendente salen **al final**. Si no es lo que quieres, hay
variantes explícitas: `asc_nulls_last()` y `desc_nulls_first()`.

In [ ]:
sales.sort(F.col("Item_Weight").asc()).select("Item_Identifier", "Item_Weight").show(3)
sales.sort(F.col("Item_Weight").asc_nulls_last()).select("Item_Identifier", "Item_Weight").show(3)

## Tarea 2 · Los N primeros

`limit(n)` recorta un DataFrame a `n` filas. Combinado con `sort`, da "los N mayores" o
"los N menores". Sin `sort`, las filas que devuelve son las que primero encuentra, sin
ninguna garantía.

Crea `top_prices` con los **3 productos más caros** según `Item_MRP`.

In [ ]:
top_prices = sales.sort(F.col("Item_MRP").desc()).limit(3)

In [ ]:
check(
    "Tarea 2 · exactamente 3 filas",
    lambda: top_prices.count() == 3,
    "Termina con .limit(3).",
)
check(
    "Tarea 2 · son los precios más altos",
    lambda: [row["Item_MRP"] for row in top_prices.collect()] == [266.8884, 266.8884, 266.6884],
    'Ordena antes de recortar: sales.sort(F.col("Item_MRP").desc()).limit(3). El precio máximo es 266.8884.',
)

In [ ]:
top_prices.select("Item_Identifier", "Item_Type", "Item_MRP").show()

## Tarea 3 · Quitar columnas

`drop` elimina una o varias columnas, separadas por comas. Es el contrario de `select`:
en vez de decir cuáles quieres, dices cuáles sobran.

Crea `slim` a partir de `sales` sin las columnas `Item_Visibility` e `Item_Type`.

In [ ]:
slim = sales.drop("Item_Visibility", "Item_Type")

In [ ]:
check(
    "Tarea 3 · quedan 10 columnas, sin las dos indicadas",
    lambda: len(slim.columns) == 10 and "Item_Visibility" not in slim.columns and "Item_Type" not in slim.columns,
    'Usa sales.drop("Item_Visibility", "Item_Type"), las dos en la misma llamada.',
)
check(
    "Tarea 3 · el resto sigue en su orden",
    lambda: slim.columns == [column for column in sales.columns if column not in ("Item_Visibility", "Item_Type")],
    "drop no reordena las columnas que quedan. Si usaste select, revisa el orden.",
)

Como `withColumnRenamed`, `drop` **no avisa** si la columna no existe: devuelve el
DataFrame tal cual. Comprueba `df.columns` después de un `drop` importante.

In [ ]:
print(len(sales.drop("Item_Visibilty").columns))  # Typo on purpose.

## Tarea 4 · Duplicados

BigMart no tiene filas totalmente repetidas, así que para practicar hay un DataFrame
pequeño de pedidos con duplicados de verdad:

In [ ]:
orders = spark.createDataFrame(
    [
        (1, "ana", "Dairy", 12.5),
        (2, "luis", "Meat", 30.0),
        (2, "luis", "Meat", 30.0),
        (3, "ana", "Dairy", 8.0),
        (4, "marta", "Snack Foods", 5.5),
        (4, "marta", "Snack Foods", 5.5),
        (4, "marta", "Snack Foods", 5.5),
        (5, "luis", "Dairy", 12.5),
    ],
    "order_id INT, customer STRING, item_type STRING, amount DOUBLE",
)
orders.show()

- `dropDuplicates()` sin argumentos quita las filas en las que **todas** las columnas
  coinciden. `distinct()` hace exactamente lo mismo.
- `dropDuplicates(["col1", "col2"])` compara solo esas columnas y se queda con **una**
  fila por combinación, con el resto de columnas de alguna de las repetidas.

Crea:

- `unique_orders`: `orders` sin filas repetidas
- `one_per_customer`: una sola fila por `customer`
- `item_types`: un DataFrame con una sola columna, `Item_Type`, y los valores distintos
  que hay en `sales`

In [ ]:
unique_orders = orders.dropDuplicates()
one_per_customer = orders.dropDuplicates(["customer"])
item_types = sales.select("Item_Type").distinct()

In [ ]:
check(
    "Tarea 4 · unique_orders tiene 5 pedidos",
    lambda: unique_orders.count() == 5 and unique_orders.columns == orders.columns,
    "Usa orders.dropDuplicates() o orders.distinct(), sin argumentos. Hay 3 filas repetidas.",
)
check(
    "Tarea 4 · one_per_customer tiene 3 filas, una por cliente",
    lambda: one_per_customer.count() == 3
    and sorted(row["customer"] for row in one_per_customer.collect()) == ["ana", "luis", "marta"],
    'Usa orders.dropDuplicates(["customer"]). Hay 3 clientes.',
)
check(
    "Tarea 4 · item_types tiene 16 tipos en una columna",
    lambda: item_types.columns == ["Item_Type"] and item_types.count() == 16,
    'Selecciona primero la columna y luego quita repetidos: sales.select("Item_Type").distinct().',
)

Un aviso sobre `dropDuplicates` con subset: Spark se queda con **una** de las filas
repetidas, pero **no garantiza cuál**. Si necesitas la más reciente o la más cara,
el módulo de window functions enseña cómo elegirla. Mira cuál ha sobrevivido aquí:

In [ ]:
one_per_customer.sort("customer").show()

## Tarea 5 · Quitar filas con nulos

`dropna` elimina filas con nulos. Tiene dos mandos:

- `how`: `"any"` (por defecto) quita la fila si **alguna** columna es nula; `"all"` solo
  si **todas** lo son.
- `subset`: la lista de columnas que se miran. Sin él, se miran todas.

En `sales` hay nulos en dos columnas: `Item_Weight` (1463) y `Outlet_Size` (2410), y
ninguna fila los tiene en las dos a la vez.

Crea:

- `with_size`: `sales` sin las filas que no tienen `Outlet_Size`
- `complete`: `sales` solo con las filas que no tienen **ningún** nulo

In [ ]:
with_size = sales.dropna(subset=["Outlet_Size"])
complete = sales.dropna(how="any")

In [ ]:
check(
    "Tarea 5 · with_size tiene 6113 filas",
    lambda: with_size.count() == 6113,
    'Usa sales.dropna(subset=["Outlet_Size"]). 8523 − 2410 = 6113.',
)
check(
    "Tarea 5 · with_size conserva los pesos nulos",
    lambda: with_size.filter(F.col("Item_Weight").isNull()).count() == 1463,
    "Con subset solo se mira Outlet_Size: los pesos nulos tienen que seguir ahí.",
)
check(
    "Tarea 5 · complete tiene 4650 filas sin ningún nulo",
    lambda: complete.count() == 4650,
    'Usa sales.dropna() o sales.dropna(how="any"). 8523 − 1463 − 2410 = 4650.',
)

`how="all"` es mucho menos agresivo: solo quita filas totalmente vacías, y en BigMart no
hay ninguna.

In [ ]:
print('how="any":', sales.dropna(how="any").count())
print('how="all":', sales.dropna(how="all").count())
print("subset=['Outlet_Size']:", sales.dropna(subset=["Outlet_Size"]).count())

## Tarea 6 · Rellenar nulos

`fillna(valor)` sustituye los nulos por un valor. Tiene una regla que sorprende: **solo
rellena las columnas cuyo tipo encaja con el valor**. Un texto rellena las columnas de
texto; un número, las numéricas. Las demás se quedan como estaban, sin aviso.

Con un diccionario se da un valor distinto a cada columna, y eso evita el problema.

Crea `filled` a partir de `sales` con `Outlet_Size` rellenado con `"Unknown"` e
`Item_Weight` rellenado con `12.86`, la media de los pesos conocidos.

In [ ]:
filled = sales.fillna({"Outlet_Size": "Unknown", "Item_Weight": 12.86})

In [ ]:
check(
    "Tarea 6 · no quedan nulos en Outlet_Size ni en Item_Weight",
    lambda: filled.filter(F.col("Outlet_Size").isNull() | F.col("Item_Weight").isNull()).count() == 0,
    'Pasa un diccionario: sales.fillna({"Outlet_Size": "Unknown", "Item_Weight": 12.86}).',
)
check(
    "Tarea 6 · 2410 tiendas marcadas como Unknown",
    lambda: filled.filter(F.col("Outlet_Size") == "Unknown").count() == 2410,
    'El valor para Outlet_Size es exactamente "Unknown".',
)
check(
    "Tarea 6 · 1463 pesos valen 12.86 y la columna sigue siendo double",
    lambda: filled.filter(F.col("Item_Weight") == 12.86).count() == 1463 and dict(filled.dtypes)["Item_Weight"] == "double",
    "El valor para Item_Weight es 12.86. fillna no cambia el tipo de la columna.",
)
check(
    "Tarea 6 · siguen las 8523 filas",
    lambda: filled.count() == 8523,
    "fillna rellena, no borra filas.",
)

Comprueba la regla del tipo. Un solo valor de texto rellena `Outlet_Size`, pero deja
`Item_Weight` con sus 1463 nulos; y pedir explícitamente que rellene `Item_Weight` con un
texto tampoco hace nada:

In [ ]:
text_fill = sales.fillna("Not Available")
print("fillna('Not Available') · Outlet_Size nulos:", text_fill.filter(F.col("Outlet_Size").isNull()).count())
print("fillna('Not Available') · Item_Weight nulos:", text_fill.filter(F.col("Item_Weight").isNull()).count())

forced = sales.fillna("Not Available", subset=["Item_Weight"])
print("fillna('Not Available', subset=['Item_Weight']) · Item_Weight nulos:", forced.filter(F.col("Item_Weight").isNull()).count())

## Para terminar · dropna frente a fillna

Las dos opciones quitan los nulos de en medio, pero no son intercambiables. `dropna`
pierde filas: aquí, el 45 % de los datos si quitas cualquier nulo. `fillna` las conserva,
pero inventa un valor, y ese valor contamina las medias y los conteos que hagas después.
Antes de elegir, mira cuántos nulos hay y en qué columnas:

In [ ]:
null_counts = sales.select([F.count(F.when(F.col(column).isNull(), 1)).alias(column) for column in sales.columns])
null_counts.show(vertical=True)

## Resumen

Ejecuta esta celda al final. Si todos los tests pasan, vuelve al sitio y marca el
laboratorio como completado.

In [ ]:
EXPECTED_TESTS = 17
passed_tests = sum(RESULTS.values())

print(f"{passed_tests} de {EXPECTED_TESTS} tests superados")
for test_name, passed in RESULTS.items():
    print("  ✅" if passed else "  ❌", test_name)

assert passed_tests == EXPECTED_TESTS, "Todavía quedan tareas por resolver."
print("\n🎉 ¡Laboratorio completado! Vuelve al sitio y márcalo como hecho.")